<a href="https://colab.research.google.com/github/clepigdo/House_Price_Prediction/blob/main/EDA_House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/FASTTRACK_BENGKOD/Raw-Final-Rumah.csv')

In [ ]:
df.head()

,Harga,Kamar Tidur,Kamar Mandi,Garasi,Luas Tanah,Luas Bangunan,Kecamatan,Page
0,"2,5 Miliar",4,3,10,LT : 438 m²,LB : 110 m²,Tapos,129
1,385 Juta,3,2,1,LT : 50 m²,LB : 60 m²,Pancoran Mas,129
2,385 Juta,3,2,1,LT : 50 m²,LB : 60 m²,Cipayung,129
3,385 Juta,3,2,1,LT : 50 m²,LB : 60 m²,Rangkapanjaya,129
4,385 Juta,3,2,1,LT : 50 m²,LB : 60 m²,Pancoran Mas,129


In [ ]:
df.shape

(40200, 8)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40200 entries, 0 to 40199
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Harga          40200 non-null  object
 1   Kamar Tidur    40191 non-null  object
 2   Kamar Mandi    35797 non-null  object
 3   Garasi         35011 non-null  object
 4   Luas Tanah     32501 non-null  object
 5   Luas Bangunan  24598 non-null  object
 6   Kecamatan      40200 non-null  object
 7   Page           40200 non-null  int64 
dtypes: int64(1), object(7)
memory usage: 2.5+ MB


In [ ]:
df.isnull().sum()

,0
Harga,0
Kamar Tidur,9
Kamar Mandi,4403
Garasi,5189
Luas Tanah,7699
Luas Bangunan,15602
Kecamatan,0
Page,0


selanjutnya kita akan cek missing value yang paling banyak dulu yaitu Luas Bangunan

In [ ]:

df[df['Luas Bangunan'].isnull()].head(10)

,Harga,Kamar Tidur,Kamar Mandi,Garasi,Luas Tanah,Luas Bangunan,Kecamatan,Page
7,525 Juta,2,1,LB:41 m²,NaN,NaN,Cinere,129
8,"2,5 Miliar",3,2,LT:409 m²,LB : 300 m²,NaN,Sukmajaya,129
9,"3,7 Miliar",4,3,LT:280 m²,LB : 240 m²,NaN,Cinere,129
11,"5,3 Miliar Total",LT:1368 m²,NaN,NaN,NaN,NaN,Depok I,129
13,525 Juta,1,1,LB:25 m²,NaN,NaN,Margonda,129
14,"2,18 Miliar",3,3,LT:128 m²,LB : 138 m²,NaN,Cibubur,129
16,30 Miliar Total,LT:2520 m²,NaN,NaN,NaN,NaN,Depok II,129
17,"1,7 Miliar",3,2,LB:73 m²,NaN,NaN,Cibubur,129
26,"28,1 Juta Total",LT:12 m²,NaN,NaN,NaN,NaN,Depok I,130
28,40 Miliar,LT:4000 m²,LB:1200 m²,NaN,NaN,NaN,Margonda,130


pada bagian Luas bangunan dan Luas Tanah ini terdapat keanehan beberapa data malah masuk ke kamar tidur, kamar mandi, garasi, ini bisa terjadi mungkin saja karna kesalahan scraping data/kesalahan dari webnya

In [ ]:
kita_periksa_kolom = [
    'Kamar Tidur',
    'Kamar Mandi',
    'Garasi',
    'Luas Bangunan',
    'Luas Tanah'
]
for kolom in kita_periksa_kolom:
    jumlah_lt = df[kolom].astype('string').str.contains(
        r'\bLT\s*:', case=False, na=False
    ).sum()

    jumlah_lb = df[kolom].astype('string').str.contains(
        r'\bLB\s*:', case=False, na=False
    ).sum()

    print(f'{kolom}: LT = {jumlah_lt}, LB = {jumlah_lb}')

Kamar Tidur: LT = 4851, LB = 90
Kamar Mandi: LT = 1379, LB = 782
Garasi: LT = 7906, LB = 2498
Luas Bangunan: LT = 0, LB = 24598
Luas Tanah: LT = 24600, LB = 7901


In [ ]:
# Memeriksa keberadaan LB di setiap kolom terkait
ada_lb = df[kita_periksa_kolom].astype('string').apply(
    lambda kolom: kolom.str.contains(
        r'\bLB\s*:', case=False, na=False
    )
)

# Menghitung baris yang memiliki setidaknya satu LB
jumlah_baris_lb = ada_lb.any(axis=1).sum()

# Menghitung baris yang tidak memiliki label LB
jumlah_tanpa_lb = len(df) - jumlah_baris_lb

print("Baris dengan label LB:", jumlah_baris_lb)
print("Baris tanpa label LB:", jumlah_tanpa_lb)


Baris dengan label LB: 35869
Baris tanpa label LB: 4331


In [ ]:

# Baris dengan kolom Luas Bangunan kosong
lb_kosong = df['Luas Bangunan'].isnull()

# Baris yang kolom Luas Bangunannya kosong,
# tetapi memiliki label LB di kolom lain
lb_terdeteksi = lb_kosong & ada_lb.any(axis=1)

print("Luas Bangunan kosong:", lb_kosong.sum())
print("Kosong tetapi memiliki label LB:", lb_terdeteksi.sum())
print("Kosong dan tidak ditemukan label LB:",
      (lb_kosong & ~ada_lb.any(axis=1)).sum())


Luas Bangunan kosong: 15602
Kosong tetapi memiliki label LB: 11271
Kosong dan tidak ditemukan label LB: 4331


In [ ]:

# Menampilkan 10 baris yang berpotensi dipulihkan
df.loc[lb_terdeteksi, [
    'Kamar Tidur',
    'Kamar Mandi',
    'Garasi',
    'Luas Tanah',
    'Luas Bangunan'
]].head(10)


,Kamar Tidur,Kamar Mandi,Garasi,Luas Tanah,Luas Bangunan
7,2,1,LB:41 m²,NaN,NaN
8,3,2,LT:409 m²,LB : 300 m²,NaN
9,4,3,LT:280 m²,LB : 240 m²,NaN
13,1,1,LB:25 m²,NaN,NaN
14,3,3,LT:128 m²,LB : 138 m²,NaN
17,3,2,LB:73 m²,NaN,NaN
28,LT:4000 m²,LB:1200 m²,NaN,NaN,NaN
30,4,LT:1050 m²,LB:650 m²,NaN,NaN
35,2,LT:100 m²,LB:106 m²,NaN,NaN
36,3,2,LT:49 m²,LB : 98 m²,NaN


disetiap baris yang tampil ini terdapat LB di dalamnya, entah di fitur kamar tidur sampai luas tanah, dan dari 10 yang tampil ini hanya memiliki masing masing 1 LB di setiap baris

In [ ]:
# Menghitung berapa kolom yang memiliki label LB pada setiap baris
jumlah_lb_per_baris = ada_lb.sum(axis=1)

# Melihat distribusi jumlah label LB
jumlah_lb_per_baris.value_counts().sort_index()

,count
0,4331
1,35869


disini terlihat ada 4331 data tidak terdapat LB di barisnya, dan terdapat 35869 data setidaknya di 1 kolom

In [ ]:
# Mengumpulkan nilai yang memiliki label LB dari setiap kolom
contoh_lb = []

for kolom in kita_periksa_kolom:
    nilai_lb = df.loc[ada_lb[kolom], kolom]
    contoh_lb.extend(nilai_lb.tolist())

# Menampilkan 20 contoh nilai LB yang berbeda
pd.Series(contoh_lb).drop_duplicates().head(20)


,0
0,LB:27 m²
1,LB:22 m²
3,LB:21 m²
4,LB:30 m²
5,LB:24 m²
6,LB:20 m²
7,LB:25 m²
14,LB:51 m²
15,LB:17 m²
16,LB:23 m²


In [ ]:

# Menggabungkan semua nilai LB yang ditemukan
semua_lb = pd.Series(contoh_lb, dtype='string')

# Pola format LB yang diharapkan
pola_lb = r'^\s*LB\s*:\s*\d+(?:[.,]\d+)?\s*m²\s*$'

# Memeriksa nilai yang tidak sesuai format
format_tidak_sesuai = ~semua_lb.str.fullmatch(
    pola_lb, case=False, na=False
)

print("Total nilai berlabel LB:", len(semua_lb))
print("Format sesuai:", (~format_tidak_sesuai).sum())
print("Format tidak sesuai:", format_tidak_sesuai.sum())

Total nilai berlabel LB: 35869
Format sesuai: 35869
Format tidak sesuai: 0


SELANJUTNYA KITA BERALIH KE LT, KARNA TADI ADA DISINGGUNG KETIKA NYARI LB ADA DATA LT JUGA SAMA MISSPLACED

In [ ]:
# Memeriksa label LT pada seluruh kolom terkait
ada_lt = pd.DataFrame({
    kolom: df[kolom].astype('string').str.contains(
        r'\bLT\s*:', case=False, na=False
    )
    for kolom in kita_periksa_kolom
})

# Menghitung jumlah kolom yang mengandung LT per baris
jumlah_lt_per_baris = ada_lt.sum(axis=1)

# Menampilkan distribusinya
jumlah_lt_per_baris.value_counts().sort_index()


,count
0,1464
1,38736


terlihat sebanyak 1464 LT tidak ada di barisnya dan 38736 setidaknya di satu kolom

In [ ]:

# Mengumpulkan nilai yang mengandung label LT
semua_lt = []

for kolom in kita_periksa_kolom:
    nilai_lt = df.loc[ada_lt[kolom], kolom]
    semua_lt.extend(nilai_lt.tolist())

# Mengubah menjadi Series
semua_lt = pd.Series(semua_lt, dtype='string')

# Pola format LT yang diharapkan
pola_lt = r'\s*LT\s*:\s*\d+(?:[.,]\d+)?\s*m²\s*'

# Memeriksa konsistensi format
format_sesuai = semua_lt.str.fullmatch(
    pola_lt, case=False, na=False
)

print("Total nilai berlabel LT:", len(semua_lt))
print("Format sesuai:", format_sesuai.sum())
print("Format tidak sesuai:", (~format_sesuai).sum())


Total nilai berlabel LT: 38736
Format sesuai: 38736
Format tidak sesuai: 0


In [ ]:

# Baris dengan kolom Luas Tanah kosong
lt_kosong = df['Luas Tanah'].isnull()

# Baris yang memiliki label LT di salah satu kolom
lt_ditemukan = ada_lt.any(axis=1)

# Menghitung kondisi informasi LT
print("Luas Tanah kosong:", lt_kosong.sum())

print("Kosong tetapi label LT ditemukan:",
      (lt_kosong & lt_ditemukan).sum())

print("Kosong dan label LT tidak ditemukan:",
      (lt_kosong & ~lt_ditemukan).sum())


Luas Tanah kosong: 7699
Kosong tetapi label LT ditemukan: 6236
Kosong dan label LT tidak ditemukan: 1463


In [ ]:

# Mengambil semua baris dengan Luas Tanah kosong
# dan label LT tidak ditemukan
lt_belum_ditemukan = df.loc[
    lt_kosong & ~lt_ditemukan
].copy()

# Memeriksa kondisi semua kolom
hasil_audit = pd.DataFrame({
    'Terisi': lt_belum_ditemukan.notna().sum(),
    'Kosong': lt_belum_ditemukan.isna().sum()
})

# Menghitung persentase nilai terisi
hasil_audit['Persentase Terisi (%)'] = (
    hasil_audit['Terisi'] / len(lt_belum_ditemukan) * 100
).round(2)

print("Total baris diperiksa:", len(lt_belum_ditemukan))

display(hasil_audit)

Total baris diperiksa: 1463


,Terisi,Kosong,Persentase Terisi (%)
Harga,1463,0,100.00
Kamar Tidur,1454,9,99.38
Kamar Mandi,1363,100,93.16
Garasi,1125,338,76.90
Luas Tanah,0,1463,0.00
Luas Bangunan,0,1463,0.00
Kecamatan,1463,0,100.00
Page,1463,0,100.00


In [ ]:

# Mengambil hasil pemeriksaan label LB
# khusus untuk 1.463 baris
audit_lb = ada_lb.loc[lt_belum_ditemukan.index]

print("Total baris:", len(audit_lb))

print("Memiliki label LB:",
      audit_lb.any(axis=1).sum())

print("Tidak memiliki label LB:",
      (~audit_lb.any(axis=1)).sum())


Total baris: 1463
Memiliki label LB: 1443
Tidak memiliki label LB: 20


In [ ]:
# Memilih baris tanpa label LT maupun LB
tanpa_lt_lb = lt_belum_ditemukan.loc[
    ~audit_lb.any(axis=1)
]

# Menampilkan seluruh baris
display(tanpa_lt_lb)

,Harga,Kamar Tidur,Kamar Mandi,Garasi,Luas Tanah,Luas Bangunan,Kecamatan,Page
1977,"5,3 Juta /m²",NaN,NaN,NaN,NaN,NaN,Sukmajaya,62
3622,6 Miliar Total,NaN,NaN,NaN,NaN,NaN,Cinere,1114
8801,7 Juta /m²,NaN,NaN,NaN,NaN,NaN,Cinere,1373
9117,700 Miliar,4,3,NaN,NaN,NaN,Pangkalan Jati,1388
12750,"3,54 Miliar",NaN,NaN,NaN,NaN,NaN,Cilodong,1570
15161,20 Miliar,4,3,NaN,NaN,NaN,Pangkalan Jati,1691
15406,"2,75 Miliar",3,4,1,NaN,NaN,Cinere,1703
16827,675 Juta,2,1,NaN,NaN,NaN,Cimanggis,1774
18216,9 Miliar,7,7,2,NaN,NaN,Cinere,1843
18821,950 Juta,3,2,1,NaN,NaN,Cilodong,1874


In [ ]:

# Memeriksa pola penulisan harga
harga = df['Harga'].astype('string')

harga_per_meter = harga.str.contains(
    r'/\s*m(?:²|2)', case=False, na=False
)

harga_total = harga.str.contains(
    r'\bTotal\b', case=False, na=False
)

print("Harga per m²:", harga_per_meter.sum())
print("Harga dengan keterangan Total:", harga_total.sum())
print("Harga tanpa kedua keterangan:",
      (~harga_per_meter & ~harga_total).sum())

print("Harga dengan kedua keterangan:",
      (harga_per_meter & harga_total).sum())


Harga per m²: 55
Harga dengan keterangan Total: 5950
Harga tanpa kedua keterangan: 34195
Harga dengan kedua keterangan: 0
